# Análise exploratória — Imóveis em São Paulo

Objetivo desta primeira etapa: **conhecer a base antes de modelar**. Vamos responder, em ordem:

1. Qual o tamanho e o formato da base?
2. Os tipos das colunas fazem sentido?
3. Há valores nulos?
4. Há linhas duplicadas?
5. A coluna `Valor` mistura coisas diferentes?
6. Há outliers impossíveis?

Cada achado vira uma decisão de limpeza, registrada no final.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv("../data/dados_imoveis.csv")
df.head(10)

,Rua,Bairro,Cidade,Metragem,Quartos,Banheiros,Vagas,Valor
0,"Avenida Itacira, 255",Planalto Paulista,São Paulo,1000,4,8,6,R$ 7.000.000
1,"Rua Aurelia Perez Alvarez, 42",Jardim dos Estados,São Paulo,496,4,4,4,R$ 3.700.000
2,Rua Alba Valdez,Jardim Reimberg,São Paulo,125,4,3,2,R$ 380.000
3,NaN,Jardim Morumbi,São Paulo,310,3,2,4,R$ 685.000
4,"Rua Tobias Barreto, 195",Mooca,São Paulo,100,3,2,2,R$ 540.000
5,Rua Graham Bell,Santo Amaro,São Paulo,440,4,4,6,R$ 1.980.000
6,"Rua Francisco Paulo, 31",Cidade Mãe do Céu,São Paulo,145,4,4,2,R$ 850.000
7,"Rua Guilherme Valente, 150",Jardim Vergueiro,São Paulo,150,2,2,2,R$ 450.000
8,"Rua Sagrada Família, 8",Jardim Benfica,São Paulo,48,2,1,1,R$ 199.900
9,Rua Tapaji,Vila Alpina,São Paulo,114,3,3,2,R$ 585.000


In [2]:
print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")

Linhas: 10,008 | Colunas: 8


In [3]:
# A base é apresentada como "Grande São Paulo", mas podemos identificar apenas um município (São Paulo) ao invés de uma região metropolitana

print(df["Cidade"].value_counts())
print("\nBairros distintos:", df["Bairro"].nunique())

Cidade
São Paulo    10008
Name: count, dtype: int64

Bairros distintos: 701


**Achado:** a base é só de São Paulo, então `Cidade` não traz informação nenhuma — pode ser descartada. São centenas de bairros.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10008 entries, 0 to 10007
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Rua        6574 non-null   str  
 1   Bairro     9996 non-null   str  
 2   Cidade     10008 non-null  str  
 3   Metragem   10008 non-null  int64
 4   Quartos    10008 non-null  int64
 5   Banheiros  10008 non-null  int64
 6   Vagas      10008 non-null  int64
 7   Valor      10008 non-null  str  
dtypes: int64(4), str(4)
memory usage: 625.6 KB


**Achado:** `Valor` aparece como texto (`object`/`str`), não como número. Antes de qualquer conta precisamos entender por quê.

In [5]:
# Amostra aleatória dos valores para ver o formato
df["Valor"].sample(15, random_state=1).tolist()

['R$ 598.000',
 'R$ 1.350.000',
 'R$ 1.200.000',
 'R$ 11.000\r\n                      /Mês',
 'R$ 1.648.000',
 'R$ 750.000',
 'R$ 8.000\r\n                      /Mês',
 'R$ 12.000.000',
 'R$ 760.000',
 'R$ 3.000.000',
 'R$ 10.000\r\n                      /Mês',
 'R$ 450.000',
 'R$ 1.080.000',
 'R$ 17.000\r\n                      /Mês',
 'R$ 650.000']

**Achado importante:** além do formato `R$ 1.350.000`, aparecem valores como `R$ 11.000 /Mês`. Ou seja, **a base mistura imóveis à venda com imóveis para aluguel**. Se não separarmos, um aluguel de R$ 8 mil seria lido como uma casa barata de R$ 8 mil.

In [6]:
# Quais sufixos de periodicidade existem? (tudo que vem depois da '/')

sufixo = df["Valor"].str.extract(r"/\s*(\w+)")[0]
print(sufixo.value_counts(dropna=False).rename({np.nan: "(sem sufixo = venda)"}))

0
(sem sufixo = venda)    9156
Mês                      848
Ano                        2
Dia                        2
Name: count, dtype: int64


In [7]:
eh_aluguel = df["Valor"].str.contains("/", na=False)
print(eh_aluguel.value_counts().rename({True: "Aluguel", False: "Venda"}))
print(f"\nAluguel = {eh_aluguel.mean():.1%} da base")

Valor
Venda      9156
Aluguel     852
Name: count, dtype: int64

Aluguel = 8.5% da base


In [8]:
# Algum valor tem formato diferente desses dois? (ex.: 'Sob consulta')
fora_do_padrao = ~df["Valor"].str.match(r"^R\$\s?[\d\.]+", na=False)
df.loc[fora_do_padrao, "Valor"].value_counts()

Series([], Name: count, dtype: int64)

In [9]:
# Converter texto -> número e marcar o tipo de negócio
def converter_valor(texto):
    if pd.isna(texto):
        return np.nan
    numero = (texto.split("/")[0]          # tira '/Mês', '/Ano', '/Dia'
                   .replace("R$", "")
                   .replace(".", "")       # separador de milhar
                   .strip())
    return pd.to_numeric(numero, errors="coerce")

df["Valor_num"] = df["Valor"].apply(converter_valor)
df["Tipo"] = np.where(eh_aluguel, "Aluguel", "Venda")
df["Periodo"] = sufixo.fillna("")  # Mês, Ano, Dia ou vazio (venda)

df.groupby("Tipo")["Valor_num"].describe()

,count,mean,std,min,25%,50%,75%,max
Tipo,,,,,,,,
Aluguel,852.00,"21,219.15","17,007.45","1,650.00","10,000.00","16,000.00","26,000.00","120,000.00"
Venda,"9,156.00","3,052,202.96","3,944,208.19","13,000.00","890,000.00","1,800,000.00","3,500,000.00","65,000,000.00"


In [10]:
# Converter texto -> número e marcar o tipo de negócio - ADICIONADO NA PREPARAÇÃO DOS DADOS
"""
def converter_valor(texto):
    if pd.isna(texto):
        return np.nan
    numero = (texto.split("/")[0]          # tira '/Mês', '/Ano', '/Dia'
                   .replace("R$", "")
                   .replace(".", "")       # separador de milhar
                   .strip())
    return pd.to_numeric(numero, errors="coerce")

df["Valor_num"] = df["Valor"].apply(converter_valor)
df["Tipo"] = np.where(eh_aluguel, "Aluguel", "Venda")
df["Periodo"] = sufixo.fillna("")  # Mês, Ano, Dia ou vazio (venda)

df.groupby("Tipo")["Valor_num"].describe()
"""

'\ndef converter_valor(texto):\n    if pd.isna(texto):\n        return np.nan\n    numero = (texto.split("/")[0]          # tira \'/Mês\', \'/Ano\', \'/Dia\'\n                   .replace("R$", "")\n                   .replace(".", "")       # separador de milhar\n                   .strip())\n    return pd.to_numeric(numero, errors="coerce")\n\ndf["Valor_num"] = df["Valor"].apply(converter_valor)\ndf["Tipo"] = np.where(eh_aluguel, "Aluguel", "Venda")\ndf["Periodo"] = sufixo.fillna("")  # Mês, Ano, Dia ou vazio (venda)\n\ndf.groupby("Tipo")["Valor_num"].describe()\n'

Existe uma diferença de escala entre os 2 grupos: são populações diferentes e devem ser analisadas separadamente.

# **Auditoria da base de dados**

Fazer a auditoria dos dados, verificando duplicatas e valores nulos é importante para garantir a qualidade da análise. A seguir, verificamos a quantidade de linhas duplicadas no DataFrame, considerando todas as colunas e também apenas a coluna 'id', que foi criada para identificar registros únicos.

In [11]:
import re
import hashlib
import pandas as pd

In [12]:
# Criando coluna id de cada registro com as seguintes informações: Rua(Iniciais+numero)+Bairo(Iniciais)+Cidade(Iniciais)+metragem+quartos+banheiros+vagas
df.insert(0, 'Id', df.apply(lambda r: f"{''.join([p[0].lower() for p in (re.sub(r'\d+', '', str(r['Rua'])).split() if pd.notna(r['Rua']) else ['x']) if p.lower() not in ['da','de','do','dos','o','a']])}{''.join(filter(str.isdigit, str(r['Rua']))) if pd.notna(r['Rua']) and any(c.isdigit() for c in str(r['Rua'])) else '0'}_{''.join([p[0].lower() for p in str(r['Bairro']).split() if p.lower() not in ['da','de','do','dos','o','a']]) if pd.notna(r['Bairro']) else 'x'}_{''.join([p[0].lower() for p in str(r['Cidade']).split() if p.lower() not in ['da','de','doên', 'dos','o','a']]) if pd.notna(r['Cidade']) else 'x'}_{int(r['Metragem'])}_{int(r['Quartos'])}_{int(r['Banheiros'])}_{int(r['Vagas'])}", axis=1))

df.head(10)

# salvando dataframe com a coluna id
#df.to_csv("../data/dados_imoveis_com_id.csv", index=False)


,Id,Rua,Bairro,Cidade,Metragem,Quartos,Banheiros,Vagas,Valor,Valor_num,Tipo,Periodo
0,ai255_pp_sp_1000_4_8_6,"Avenida Itacira, 255",Planalto Paulista,São Paulo,1000,4,8,6,R$ 7.000.000,7000000,Venda,
1,rapa42_je_sp_496_4_4_4,"Rua Aurelia Perez Alvarez, 42",Jardim dos Estados,São Paulo,496,4,4,4,R$ 3.700.000,3700000,Venda,
2,rav0_jr_sp_125_4_3_2,Rua Alba Valdez,Jardim Reimberg,São Paulo,125,4,3,2,R$ 380.000,380000,Venda,
3,x0_jm_sp_310_3_2_4,NaN,Jardim Morumbi,São Paulo,310,3,2,4,R$ 685.000,685000,Venda,
4,rtb195_m_sp_100_3_2_2,"Rua Tobias Barreto, 195",Mooca,São Paulo,100,3,2,2,R$ 540.000,540000,Venda,
5,rgb0_sa_sp_440_4_4_6,Rua Graham Bell,Santo Amaro,São Paulo,440,4,4,6,R$ 1.980.000,1980000,Venda,
6,rfp31_cmc_sp_145_4_4_2,"Rua Francisco Paulo, 31",Cidade Mãe do Céu,São Paulo,145,4,4,2,R$ 850.000,850000,Venda,
7,rgv150_jv_sp_150_2_2_2,"Rua Guilherme Valente, 150",Jardim Vergueiro,São Paulo,150,2,2,2,R$ 450.000,450000,Venda,
8,rsf8_jb_sp_48_2_1_1,"Rua Sagrada Família, 8",Jardim Benfica,São Paulo,48,2,1,1,R$ 199.900,199900,Venda,
9,rt0_va_sp_114_3_3_2,Rua Tapaji,Vila Alpina,São Paulo,114,3,3,2,R$ 585.000,585000,Venda,


### **1. TIPOS: Verificar se cada coluna tem o tipo esperado**

In [13]:
print("\n=== 1. TIPOS DE DADOS ===")
print("\nTipos de dados por coluna:")
print(df.info())


=== 1. TIPOS DE DADOS ===

Tipos de dados por coluna:
<class 'pandas.DataFrame'>
RangeIndex: 10008 entries, 0 to 10007
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Id         10008 non-null  str  
 1   Rua        6574 non-null   str  
 2   Bairro     9996 non-null   str  
 3   Cidade     10008 non-null  str  
 4   Metragem   10008 non-null  int64
 5   Quartos    10008 non-null  int64
 6   Banheiros  10008 non-null  int64
 7   Vagas      10008 non-null  int64
 8   Valor      10008 non-null  str  
 9   Valor_num  10008 non-null  int64
 10  Tipo       10008 non-null  str  
 11  Periodo    10008 non-null  str  
dtypes: int64(5), str(7)
memory usage: 938.4 KB
None


### **2. FALTANTES: Valores vazios ou disfarçados**

In [14]:
print("NaNs nativos por coluna:")
print(df.isna().sum())

NaNs nativos por coluna:
Id              0
Rua          3434
Bairro         12
Cidade          0
Metragem        0
Quartos         0
Banheiros       0
Vagas           0
Valor           0
Valor_num       0
Tipo            0
Periodo         0
dtype: int64


### **3. VALORES POSSÍVEIS: Faixas plausíveis e outliers**

In [15]:
import plotly.express as px

In [16]:
print("============= Resumo estatístico das colunas numéricas ===============")
print(df['Valor_num'].describe())

============= Resumo estatístico das colunas numéricas ===============
count       10,008.00
mean     2,794,169.56
std      3,866,248.54
min          1,650.00
25%        750,000.00
50%      1,600,000.00
75%      3,350,000.00
max     65,000,000.00
Name: Valor_num, dtype: float64


### **4. DUPLICATAS: Linhas ou IDs repetidos**

In [17]:
print("Quantidade de linhas duplicadas: ", df.duplicated().sum())
print("Quantidade de linhas duplicadas considerando apenas a coluna 'Id': ", df.duplicated(subset=['Id']).sum())

ids_afetados = df[df.duplicated(subset=["Id"], keep=False)]["Id"].unique()
print(f"Ids afetados: {ids_afetados}")

Quantidade de linhas duplicadas:  555
Quantidade de linhas duplicadas considerando apenas a coluna 'Id':  648
Ids afetados: <StringArray>
[         'x0_jg_sp_650_4_6_4',        'rma0_vr_sp_160_3_4_2',
          'x0_ja_sp_160_3_4_3', 'rdfapp0_js(o_sp_1643_4_5_10',
     'tdeao0_vnc_sp_170_3_5_3',         'ri0_jp_sp_136_3_4_3',
       'rc652_vm_sp_192_3_3_2',          'aa0_i_sp_218_3_3_3',
          'x0_je_sp_525_4_6_4',         'aa0_cj_sp_840_4_6_8',
 ...
        'rel0_pi_sp_125_6_3_3',          'x0_jp_sp_423_5_7_6',
          'rb0_p_sp_439_2_3_2',        'rpt0_vo_sp_157_3_2_2',
         'ao0_pp_sp_267_3_6_4',          'rt0_vs_sp_95_2_2_2',
         'rg0_bs_sp_605_8_7_5',        'reb0_vs_sp_205_3_3_6',
     'rmo,514_vo_sp_185_3_2_2',        'rbt0_va_sp_122_3_2_2']
Length: 526, dtype: str


### **5. ANÁLISE DE VARIÁVEIS CATEGÓRICAS: Valores únicos disponíveis**

In [18]:
#### Verificando valores unicos das variáveis categóricas
print("\nVerificando os valores únicos das variáveis categóricas")
print(f"Cidade: {df['Cidade'].unique()}")
print(f"Tipo: {df['Tipo'].unique()}")


Verificando os valores únicos das variáveis categóricas
Cidade: <StringArray>
['São Paulo']
Length: 1, dtype: str
Tipo: <StringArray>
['Venda', 'Aluguel']
Length: 2, dtype: str


### **6. ANÁLISE DE VALORES DISCREPANTES**

**Aluguel**

In [19]:
# Detectando valores discrepantes na coluna Volume_num nos imoveis do tipo aluguel
print("Valores discrepantes dos imoveis tipo aluguel")
df_aluguel = df[df['Tipo'] == 'Aluguel']

Q1 = df_aluguel["Valor_num"].quantile(0.25)
Q3 = df_aluguel["Valor_num"].quantile(0.75)
IQR = Q3 - Q1

# Definir os limites estatísticos
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Filtrar os outliers estatísticos
outliers_iqr = df_aluguel[
    (df_aluguel["Valor_num"] < limite_inferior) | (df_aluguel["Valor_num"] > limite_superior)
]
print(f"\nTotal de imoveis (ALUGUEL): {len(df_aluguel)}")
print(f"Outliers: {len(outliers_iqr)}")

print(f"Limite Inferior: {limite_inferior} | Limite Superior: {limite_superior}")
print("\nOutliers encontrados pelo IQR:")
print(outliers_iqr[["Id", "Valor_num"]])

Valores discrepantes dos imoveis tipo aluguel

Total de imoveis (ALUGUEL): 852
Outliers: 43
Limite Inferior: -14000.0 | Limite Superior: 50000.0

Outliers encontrados pelo IQR:
                            Id  Valor_num
191         x0_m_sp_1800_3_5_6      80000
924       rm0_cma_sp_800_4_6_5      65000
959     rplo0_m_sp_1100_4_9_18      60000
1012        x0_ja_sp_850_5_5_6      70000
1020      ra0_ja_sp_880_3_5_10      65000
1104   rteum0_fm_sp_1500_4_5_7      60000
1218     rtf0_sa_sp_800_6_10_8      55000
1815       ra0_ja_sp_880_4_8_8      65000
1859      ra0_ja_sp_860_3_6_13      65000
1906       ra0_ja_sp_885_4_8_8      65000
1961      ra0_ja_sp_865_3_6_13      65000
2039       x0_je_sp_1250_4_4_6      64000
2101       x0_jl_sp_1400_5_9_5      55000
2104       x0_jl_sp_1405_5_9_5      55000
2206       x0_jl_sp_1405_5_9_5      55000
2491      x0_ja_sp_1141_5_5_10     100000
2513     x0_vt_sp_1650_6_16_42     100000
2739      x0_cj_sp_1530_5_5_30      90000
2769      x0_cj_sp_1535_5

**Venda**

In [20]:
# Detectando valores discrepantes na coluna Volume_num nos imoveis do tipo aluguel
print("Valores discrepantes dos imoveis tipo aluguel")
df_venda = df[df['Tipo'] == 'Venda']

Q1 = df_venda["Valor_num"].quantile(0.25)
Q3 = df_venda["Valor_num"].quantile(0.75)
IQR = Q3 - Q1

# Definir os limites estatísticos
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Filtrar os outliers estatísticos
outliers_iqr = df_venda[
    (df_venda["Valor_num"] < limite_inferior) | (df_venda["Valor_num"] > limite_superior)
]
print(f"\nTotal de imoveis (ALUGUEL): {len(df_venda)}")
print(f"Outliers: {len(outliers_iqr)}")

print(f"Limite Inferior: {limite_inferior} | Limite Superior: {limite_superior}")
print("\nOutliers encontrados pelo IQR:")
print(outliers_iqr[["Id", "Valor_num"]])

Valores discrepantes dos imoveis tipo aluguel

Total de imoveis (ALUGUEL): 9156
Outliers: 778
Limite Inferior: -3025000.0 | Limite Superior: 7415000.0

Outliers encontrados pelo IQR:
                         Id  Valor_num
30     rp0_cj_sp_1195_4_4_6    8700000
67      rb0_ja_sp_663_4_7_6   14500000
68    rgft0_jp_sp_420_4_6_3    8700000
102    ri0_je_sp_756_5_4_10   37000000
108     x0_jp_sp_1060_5_5_4   30000000
...                     ...        ...
9923    x0_jl_sp_1345_5_9_7    9500000
9948   ras0_jg_sp_605_5_7_5    8000000
9959    x0_jl_sp_1350_5_9_7    9500000
9984   ras0_jg_sp_605_5_7_5    8000000
9995    x0_jl_sp_1350_5_9_7    9500000

[778 rows x 2 columns]


### **7. VERIFICAÇÃO DE VALORES ÚNICOS POR COLUNA**

In [21]:
# Verificando se todas as colunas possuem valores únicos
colunas = ['Id', 'Rua', 'Bairro', 'Cidade', 'Metragem', 'Quartos', 'Banheiros', 'Vagas', 'Valor', 'Valor_num', 'Tipo', 'Periodo']

# Retorna True ou False para cada coluna
print("Detecção Valores distintos por coluna")
resultado = df[colunas].nunique() == 1
print(resultado)


print("Valores distintos por coluna")
contagem_unicos = df[colunas].nunique()
print(contagem_unicos)

Detecção Valores distintos por coluna
Id           False
Rua          False
Bairro       False
Cidade        True
Metragem     False
Quartos      False
Banheiros    False
Vagas        False
Valor        False
Valor_num    False
Tipo         False
Periodo      False
dtype: bool
Valores distintos por coluna
Id           9360
Rua          3130
Bairro        701
Cidade          1
Metragem      916
Quartos        15
Banheiros      15
Vagas          28
Valor        1041
Valor_num    1038
Tipo            2
Periodo         4
dtype: int64


## **Achados encontrados + Solução**

### **Achado 1** : Ausência de identificador unívoco (*ID*) por registro
**Descrição** : O conjunto de dados não possui uma coluna de chave primária ou identificador único para cada \
imóvel anunciado. \
**Causa** : Natureza bruta de extração via *web scraping* de portais imobiliários públicos, onde os anúncios \
são coletados como listas estáticas sem rastreabilidade de ID de origem. \
**Solução** : Geração programática de uma chave substituta (*Surrogate ID*) baseada em hash criptográfico \
(*MD5*) dos atributos estruturais e textuais do imóvel, garantindo rastreabilidade unívoca e suporte a auditorias \
no pipeline de observabilidade.

In [22]:
# ---------------------Correção - achado 1---------------------
# Criando identificadores hash para cada registro de imóveis, após o tratamento dos dados.
df_temp = df.fillna("") # Tratar nulos temporariamente para gerar a string de hash de forma consistente

# 2. Criar uma string combinando atributos essenciais do imóvel
# (Metragem + Quartos + Banheiros + Vagas + Bairro + Rua + Valor)
df_temp["String_Chave"] = (
  df_temp["Metragem"].astype(str)
  + "_"
  + df_temp["Quartos"].astype(str)
  + "_"
  + df_temp["Banheiros"].astype(str)
  + "_"
  + df_temp["Vagas"].astype(str)
  + "_"
  + df_temp["Bairro"]
  + "_"
  + df_temp["Rua"]
  + "_"
  + df_temp["Valor"] # Para pegar os minimos detalhes ('/Mês, por exemplo')
)

def gerar_hash(row_str):
  return hashlib.md5(row_str.encode("utf-8")).hexdigest()

df.insert(
    loc=df.columns.get_loc('Id') + 1,
    column='Id_hash',
    value=df_temp["String_Chave"].apply(gerar_hash),
)
del df_temp

### **Achado 2** : Duplicidade de registros identificada após a criação da chave de identificação (*ID hash*)
**Descrição** : Constatação de 555 registros com o mesmo código hash de identificação unívoca, provando a existência \
de anúncios repetidos ou coletas sobrepostas na base de dados. \
**Causa** : Processos de extração automatizada (*web scraping*) executados em múltiplos momentos ou a partir de diferentes \
fontes públicas que capturaram os mesmos anúncios imobiliários sem aplicar filtros prévios de unicidade. \
**Solução** : Utilização da chave substituta (*Surrogate ID*) gerada via hash para isolar e remover os registros duplicados \
com base no subconjunto de atributos do imóvel, assegurando a integridade analítica e evitando o viés de contagem no pipeline \
de observabilidade.

OBS: é necessário fazer a solução do achado 1 primeiro para proseguir com esse achado

In [23]:
print("Quantidade de linhas duplicadas: ", df.duplicated().sum())
print("Quantidade de linhas duplicadas considerando apenas a coluna 'Id': ", df.duplicated(subset=['Id']).sum(), "-> A coluna Id apresenta mais duplicatas, nao refletindo a realidade.")
print("Quantidade de linhas duplicadas considerando apenas a coluna 'Id_hash': ", df.duplicated(subset=['Id_hash']).sum())

ids_afetados = df[df.duplicated(subset=["Id_hash"], keep=False)]["Id"].unique()
print(f"Ids afetados: {ids_afetados}")

Quantidade de linhas duplicadas:  555
Quantidade de linhas duplicadas considerando apenas a coluna 'Id':  648 -> A coluna Id apresenta mais duplicatas, nao refletindo a realidade.
Quantidade de linhas duplicadas considerando apenas a coluna 'Id_hash':  555
Ids afetados: <StringArray>
[     'x0_je_sp_525_4_6_4',    'rf0_je_sp_957_3_7_10',
    'aab0_ap_sp_466_4_6_6',    'raf0_ap_sp_290_4_3_4',
       'x0_m_sp_655_3_3_5',   'ragg0_ap_sp_505_4_5_5',
     'rn0_vm_sp_285_4_4_6',    'rsv0_jp_sp_196_2_4_2',
    'rbc0_cb_sp_325_3_4_2',    'aae0_jl_sp_425_3_5_4',
 ...
    'rel0_pi_sp_125_6_3_3',      'x0_jp_sp_423_5_7_6',
      'rb0_p_sp_439_2_3_2',    'rpt0_vo_sp_157_3_2_2',
     'ao0_pp_sp_267_3_6_4',      'rt0_vs_sp_95_2_2_2',
     'rg0_bs_sp_605_8_7_5',    'reb0_vs_sp_205_3_3_6',
 'rmo,514_vo_sp_185_3_2_2',    'rbt0_va_sp_122_3_2_2']
Length: 441, dtype: str


In [24]:
# -----------------------SOLUÇÃO PARA REMOVER AS DUPLICATAS---------------------
# O comando abaixo diz: "Olhe a coluna Imovel_ID e mantenha apenas a primeira ocorrência (keep='first'), apagando as cópias"
df_limpo = df.drop_duplicates(subset=["Id_hash"], keep="first").copy()

print("Quantidade de linhas duplicadas: ", df_limpo.duplicated().sum())
print("Quantidade de linhas duplicadas considerando apenas a coluna 'Id_hash': ", df_limpo.duplicated(subset=['Id_hash']).sum())

Quantidade de linhas duplicadas:  0
Quantidade de linhas duplicadas considerando apenas a coluna 'Id_hash':  0


### **Achado 3** : Missing data na coluna Rua e Bairro (Análise 2)
**Descrição** : A variável rua Apresenta mais de 3400 valores ausentes. \
**Possíveis Causas** : A alta taxa de dados ausentes na variável Rua decorre de estratégias comerciais de privacidade \
para evitar contatos diretos com proprietários, da opcionalidade do campo em formulários de cadastro, de \
limitações técnicas na extração automatizada (web scraping) e da ausência de denominação oficial consolidada \
em imóveis novos ou loteamentos recentes. \
**Solução** : Substituir os valores nulos por uma string indicadora ("Desconhecida", "Não Informada" ou \
"Endereço Omitido") e criar uma variável que servirá de uma flag binária de auditoria (Rua_Ausente) para \
monitoramento.

In [ ]:
# ---------------------Correção - achado 3---------------------
df.insert(
    loc=df.columns.get_loc('Rua') + 1,
    column='Rua_Ausente',
    value=df['Rua'].isna().astype(int),
)
df["Rua"] = df["Rua"].fillna("Desconhecida")

df.insert(
    loc=df.columns.get_loc('Bairro') + 1,
    column='Bairro_Ausente',
    value=df['Bairro'].isna().astype(int),
)
df["Bairro"] = df["Bairro"].fillna("Desconhecido")

### **Achado 4** : Tipo de dados string na coluna Valor (Análise 1)
**Descrição** : A variável `Valor` possui o tipo de dados string quando deveria ser numerico. \
**Possíveis Causas** : A Variável utiliza sufixos para distinguir o valor de um aluguel do valor de venda de um imóvel. \
**Solução** : Como o valor do imóvel está em uma variável a parte (`Valor_num`), remover a variável \
`Valor`. Quanto a distinção dos valores de aluguel de uma venda, colocar as informações em uma \
variável a parte (tipo_aluguel = mensal/anual/diário).

In [ ]:
# ---------------------Correção - achado 4---------------------
def converter_valor(texto):
    
    if pd.isna(texto):
        return np.nan
    
    numero = (texto.split("/")[0]          # tira '/Mês', '/Ano', '/Dia'
                   .replace("R$", "")
                   .replace(".", "")       # separador de milhar
                   .strip())
    return pd.to_numeric(numero, errors="coerce")

df["Valor_num"] = df["Valor"].apply(converter_valor)
df["Tipo"] = np.where(eh_aluguel, "Aluguel", "Venda")
df["Periodo"] = sufixo.fillna("")  # Mês, Ano, Dia ou vazio (venda)

### **Achado 5** : Valores discrepantes tanto em imoveis do tipo venda quanto aluguel (análise 6)
**Descrição** : Presença de preços extremos inconsistentes na base, abrangendo desde valores de venda \
irrealisticamente baixos (erros de cadastro) e distorções de periodicidade em locações (diárias/anuais \
tratadas como mensais) até *outliers* legítimos de alto luxo. \
**Possíveis Causas** : Erros de preenchimento humano (omissão de dígitos nos formulários), falhas de normalização \
na extração automatizada (*web scraping*) de múltiplos portais, e mistura de diferentes regimes de \
comercialização (venda e aluguel) na mesma escala bruta. \
**Solução** : Aplicação de filtros rígidos de sanidade financeira (*hard constraints*), conversão unificada \
de locações para a base mensal, e utilização de modelos baseados em resíduos estatísticos para distinguir \
falhas sistêmicas de flutuações reais de mercado.


### **Achado 6** : Poucos Alugueis diários ou anuais em vez de mensais
**Descrição** : Pelo menos 4 registros de imóveis do tipo aluguel foram encontrados com valores por dia ou por ano,\
em vez de mensal. \
**Possíveis Causas** : Depende do objetivo financeiro do proprietário, do perfil do público e da localização \
do bem. Imóveis focados no turismo, viagens de negócios, eventos ou estadias rápidas (geralmente até 90 dias, \
conforme regulamentações como a Lei do Inquilinato brasileira) costumam ser mais rentáveis em alugueis diários \
(vide AirBnB e 'Quinto Andar'), além da Flexibilidade (O proprietário pode usar o imóvel em datas específicas ou reajustar os \
preços rapidamente conforme a alta e a baixa temporada). Já imóveis com foco em moradia fixa ou contratos (geralmente \
com duração de 12 a 30 meses no Brasil) geram renda passiva garantida todo mês, com menos necessidade de intervenção do \
proprietário. \
**Solução** : Padronizar o valor do aluguel convertendo para uma base mensal ou remover tais registros.

In [ ]:
# ---------------------Correção - achado 6---------------------
df = df[~df['Periodo'].isin(['Mês', 'Ano'])]

### **Achado 7** : Variáveis com um único valor (análise 7)
**Descrição** : A variável `Cidade` só possui um único valor para todos os registros ("São Paulo")\
**Possíveis Causas** : Base de dados filtrada por uma única cidade. \
**Solução** : Remover a variável.

In [ ]:
# --------------------Solução para Achado 7------------------------
df = df.drop('Cidade', axis=1)

# Pré-processamento dos dados

In [ ]:
# Removendo colunas desnecessárias
# Coluna Valor (Achado 2)
df = df.drop('Valor', axis=1)

# Coluna Período
df = df.drop('Periodo', axis=1)

In [30]:
# Visualização final
df.head()

,Id,Id_hash,Rua,Rua_Ausente,Bairro,Bairro_Ausente,Metragem,Quartos,Banheiros,Vagas,Valor_num,Tipo
0,ai255_pp_sp_1000_4_8_6,395895965f2f8bf7ad8abab872127fdc,"Avenida Itacira, 255",0,Planalto Paulista,0,1000,4,8,6,7000000,Venda
1,rapa42_je_sp_496_4_4_4,dbdf528aad68cd3c2107db484332f920,"Rua Aurelia Perez Alvarez, 42",0,Jardim dos Estados,0,496,4,4,4,3700000,Venda
2,rav0_jr_sp_125_4_3_2,60da5350978f3e645d754785fcfdf145,Rua Alba Valdez,0,Jardim Reimberg,0,125,4,3,2,380000,Venda
3,x0_jm_sp_310_3_2_4,eec7806e223f66935c14b0b995415b13,Desconhecida,1,Jardim Morumbi,0,310,3,2,4,685000,Venda
4,rtb195_m_sp_100_3_2_2,704fc5b190881cd584644f353896ed5e,"Rua Tobias Barreto, 195",0,Mooca,0,100,3,2,2,540000,Venda


In [31]:
# Salvando o conjunto de dados pré-processado
df.to_csv("../data/dados_imoveis-pre-processado.csv", index=False)

# **Observabilidade**

Seção incompleta.

In [32]:
!pip install --upgrade wandb

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\cesar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [33]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Treino: {len(X_tr)} clientes · Validação: {len(X_va)} clientes · {X_tr.shape[1]} colunas")
print(f"Taxa de churn na validação: {y_va.mean():.1%}")

NameError: name 'train_test_split' is not defined

In [ ]:
# Importa biblioteca weight & bias
import wandb
wandb.login()